# Studi Kasus: Sistem Pencarian Dokumen Tokoh Dunia dan Olahraga

# Upload dan Ekstrak File ZIP

In [2]:
from google.colab import files
import zipfile
import os

# Upload file zip
uploaded = files.upload()

# Ekstrak file boolean.zip
zip_filename = 'boolean.zip'
if zip_filename in uploaded:
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Berhasil mengekstrak file ZIP!")
else:
    print("File boolean.zip belum diunggah.")

Saving boolean.zip to boolean.zip
Berhasil mengekstrak file ZIP!


 # Import Library & Membaca Dokumen

In [4]:
import re
import pandas as pd

# Mencari jalur folder data hasil ekstrak
data_dir = 'data'
if not os.path.exists(data_dir) and os.path.exists('boolean/data'):
    data_dir = 'boolean/data'

documents = {}

# Membaca seluruh file .txt
for filename in os.listdir(data_dir):
    if filename.endswith('.txt'):
        filepath = os.path.join(data_dir, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read().lower()
            text = re.sub(r'[^\w\s]', '', text)
            documents[filename] = text.split()

doc_names = list(documents.keys())
print(f"Berhasil membaca {len(doc_names)} dokumen:")
print(doc_names)

Berhasil membaca 8 dokumen:
['lee quan yew.txt', 'table tennis.txt', 'badminton.txt', 'barack obama.txt', 'queen elizabeth.txt', 'baseball.txt', 'narendra modi.txt', 'shinzo abe.txt']


# Membangun Term-Document Incidence Matrix

In [6]:
# Mengambil seluruh kata unik (vocabulary)
vocabulary = sorted(list(set(word for words in documents.values() for word in words)))

# Membuat matriks 0 dan 1
matrix_data = {}
for term in vocabulary:
    matrix_data[term] = [1 if term in documents[doc] else 0 for doc in doc_names]

# Mengubah ke DataFrame Pandas
df_matrix = pd.DataFrame(matrix_data, index=doc_names).T

# Menampilkan 10 baris pertama matriks
df_matrix.head(10)

,lee quan yew.txt,table tennis.txt,badminton.txt,barack obama.txt,queen elizabeth.txt,baseball.txt,narendra modi.txt,shinzo abe.txt
12,0,0,0,0,1,0,0,1
13th,0,0,0,1,0,0,0,0
14th,0,0,0,0,0,0,1,0
16,1,0,0,0,0,0,0,0
17,1,0,0,0,0,0,1,0
1923,1,0,0,0,0,0,0,0
1926,0,1,0,0,1,0,0,0
1936,0,0,0,0,1,0,0,0
1947,0,0,0,0,1,0,0,0
1948,0,0,0,0,0,0,0,1


# Definisi Fungsi Helper Boolean Retrieval

In [7]:
def get_vector(term):
    """Mengembalikan vector boolean untuk term tertentu"""
    term = term.lower()
    if term in df_matrix.index:
        return df_matrix.loc[term].values
    else:
        return [0] * len(doc_names)

def search_boolean(vector_result):
    """Mengembalikan daftar nama dokumen berdasarkan hasil operasi bitwise"""
    matched_docs = [doc_names[i] for i, val in enumerate(vector_result) if val == 1]
    return matched_docs if matched_docs else "Tidak ada dokumen yang cocok."

# Uji Coba Query 1 (AND)

In [8]:
# Query 1: "sport" AND "played"
vec_sport = get_vector("sport")
vec_played = get_vector("played")
res1 = vec_sport & vec_played

print('Query 1: "sport" AND "played"')
print("Hasil:", search_boolean(res1))

Query 1: "sport" AND "played"
Hasil: ['table tennis.txt', 'badminton.txt', 'baseball.txt']


# Uji Coba Query 2 (OR)

In [9]:
# Query 2: "president" OR "prime"
vec_president = get_vector("president")
vec_prime = get_vector("prime")
res2 = vec_president | vec_prime

print('Query 2: "president" OR "prime"')
print("Hasil:", search_boolean(res2))

Query 2: "president" OR "prime"
Hasil: ['lee quan yew.txt', 'barack obama.txt', 'narendra modi.txt', 'shinzo abe.txt']


# Uji Coba Query 3 (AND NOT)

In [10]:
# Query 3: "sport" AND NOT "table"
vec_sport = get_vector("sport")
vec_table = get_vector("table")
res3 = vec_sport & (~vec_table + 2)

print('Query 3: "sport" AND NOT "table"')
print("Hasil:", search_boolean(res3))

Query 3: "sport" AND NOT "table"
Hasil: ['badminton.txt', 'baseball.txt']
